# Infinite Series — An Interactive Python Laboratory

**Companion notebook for Chapter 2 of _Advanced Calculus of One Variable with Python and AI_.**

An infinite series is understood through its sequence of partial sums

$$
S_N=\sum_{n=1}^{N}a_n.
$$

This notebook turns the main ideas of the chapter into controlled numerical experiments. It does **not** replace proofs: every numerical observation is paired with the hypotheses of the corresponding theorem.

### Learning goals

By the end of the notebook, you should be able to:

1. distinguish the behaviour of the terms $a_n$ from that of the partial sums $S_N$;
2. explore the Cauchy criterion through remote finite tails;
3. compare the ratio, root, comparison, condensation, integral, Raabe, and Bertrand tests;
4. compute rigorous remainder bounds;
5. visualize Cauchy products, Dirichlet cancellation, alternating-series trapping, and Riemann rearrangements;
6. choose an efficient convergence test and check its hypotheses.

Run the cells from top to bottom. Every interactive laboratory has a blue **Run** button.

## Cell 1 — Imports, display style, and reusable helpers

Numerical evidence always concerns a finite object such as $S_N$ or

$$
T_{n,m}=\sum_{k=n}^{m}a_k.
$$

It can suggest a theorem, reveal a pattern, or test a conjecture, but convergence is an infinite statement. This setup cell loads the numerical and widget libraries and defines a small amount of common styling. If a package is missing, the cell installs it once in the current notebook environment.

In [ ]:
import sys
import subprocess
import importlib


# Install only packages that are missing from the current Jupyter environment.
# Google Colab normally already contains all of them.
required_packages = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "ipywidgets": "ipywidgets",
    "IPython": "ipython",
}

for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

import math
import random
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, Markdown, HTML, clear_output
from scipy.special import zeta as hurwitz_zeta


plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {
    "blue": "#2563eb",
    "orange": "#ea580c",
    "green": "#15803d",
    "red": "#dc2626",
    "purple": "#7e22ce",
    "gray": "#475569",
}


def style_button(button, icon="play"):
    """Give every action button the same clear visual language."""
    button.button_style = "primary"
    button.icon = icon
    button.layout = widgets.Layout(width="170px")
    return button


def show_note(text, color="#eff6ff"):
    """Display a compact pedagogical note inside an output area."""
    display(
        HTML(
            f"<div style='padding:10px 14px;border-left:4px solid {COLORS['blue']};"
            f"background:{color};margin:8px 0'>{text}</div>"
        )
    )


display(
    HTML(
        "<div style='padding:12px 16px;background:#ecfdf5;border-left:5px solid #15803d'>"
        "<b>Setup complete.</b> The interactive laboratories are ready.</div>"
    )
)

## Cell 2 — Partial sums: the definition of a series

The series $\sum a_n$ converges to $S$ precisely when

$$
S_N=\sum_{n=1}^{N}a_n\longrightarrow S.
$$

The necessary condition $a_n\to0$ is obtained from $a_n=S_n-S_{n-1}$, but it is not sufficient. Compare the harmonic series, the alternating harmonic series, a geometric series, and a telescoping series. Use the parameter slider when the selected family contains $p$ or $r$.

In [ ]:
partial_family = widgets.Dropdown(
    options=[
        ("Harmonic: 1/n", "harmonic"),
        ("p-series: 1/n^p", "p_series"),
        ("Alternating p-series", "alternating"),
        ("Geometric: r^(n-1)", "geometric"),
        ("Telescoping: 1/[n(n+1)]", "telescoping"),
        ("Term-test failure: n/(n+1)", "term_failure"),
    ],
    value="harmonic",
    description="Family:",
    layout=widgets.Layout(width="360px"),
)
partial_parameter = widgets.FloatSlider(
    value=2.0,
    min=0.25,
    max=3.0,
    step=0.05,
    description="p:",
    continuous_update=False,
    readout_format=".2f",
    layout=widgets.Layout(width="360px"),
)
partial_N = widgets.IntSlider(
    value=80,
    min=5,
    max=500,
    step=5,
    description="N:",
    continuous_update=False,
    layout=widgets.Layout(width="360px"),
)
partial_button = style_button(widgets.Button(description="Run exploration"))
partial_output = widgets.Output()


def configure_partial_parameter(change=None):
    """Adapt the parameter slider to the selected family."""
    family = partial_family.value
    partial_parameter.disabled = family not in {"p_series", "alternating", "geometric"}
    if family == "geometric":
        partial_parameter.description = "r:"
        if not (-0.95 <= partial_parameter.value <= 1.20):
            partial_parameter.value = 0.60
        partial_parameter.min = -0.95
        partial_parameter.max = 1.20
        partial_parameter.step = 0.05
    else:
        partial_parameter.description = "p:"
        partial_parameter.min = 0.25
        partial_parameter.max = 3.00
        partial_parameter.step = 0.05
        if partial_parameter.value < 0.25:
            partial_parameter.value = 1.00


def partial_terms(family, n, parameter):
    """Return the first terms of one predefined series family."""
    n = np.asarray(n, dtype=float)
    if family == "harmonic":
        return 1.0 / n
    if family == "p_series":
        return n ** (-parameter)
    if family == "alternating":
        return (-1.0) ** (n.astype(int) - 1) * n ** (-parameter)
    if family == "geometric":
        return parameter ** (n.astype(int) - 1)
    if family == "telescoping":
        return 1.0 / (n * (n + 1.0))
    return n / (n + 1.0)


def run_partial_explorer(_=None):
    with partial_output:
        clear_output(wait=True)
        family = partial_family.value
        parameter = partial_parameter.value
        N = partial_N.value
        n = np.arange(1, N + 1)
        terms = partial_terms(family, n, parameter)
        sums = np.cumsum(terms)

        formulas = {
            "harmonic": r"$a_n=1/n$",
            "p_series": rf"$a_n=1/n^{{{parameter:.2f}}}$",
            "alternating": rf"$a_n=(-1)^{{n-1}}/n^{{{parameter:.2f}}}$",
            "geometric": rf"$a_n=({parameter:.2f})^{{n-1}}$",
            "telescoping": r"$a_n=1/[n(n+1)]$",
            "term_failure": r"$a_n=n/(n+1)$",
        }
        display(Markdown(f"### Selected family: {formulas[family]}"))

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        axes[0].plot(n, terms, color=COLORS["orange"], lw=1.6)
        axes[0].scatter(n[: min(N, 35)], terms[: min(N, 35)], s=18, color=COLORS["orange"])
        axes[0].axhline(0, color="black", lw=0.8)
        axes[0].set_title("Terms $a_n$")
        axes[0].set_xlabel("$n$")

        axes[1].plot(n, sums, color=COLORS["blue"], lw=2)
        axes[1].scatter([N], [sums[-1]], s=45, color=COLORS["red"], zorder=3)
        axes[1].set_title("Partial sums $S_N$")
        axes[1].set_xlabel("$N$")
        plt.tight_layout()
        plt.show()

        print(f"a_N = {terms[-1]:.10g}")
        print(f"S_N = {sums[-1]:.10g}")

        if family == "geometric" and abs(parameter) < 1:
            exact = 1.0 / (1.0 - parameter)
            print(f"Exact geometric sum = {exact:.10g}")
            print(f"Current error = {abs(exact - sums[-1]):.3e}")
        elif family == "telescoping":
            print(f"Exact sum = 1; current error = {abs(1.0 - sums[-1]):.3e}")
        elif family == "term_failure":
            show_note(r"Here $a_n\to1$, so the term test proves divergence immediately.", "#fef2f2")
        elif family == "harmonic":
            show_note(r"Although $a_n\to0$, the partial sums grow without bound. The term test is necessary, not sufficient.")
        elif family == "p_series":
            verdict = "converges" if parameter > 1 else "diverges"
            show_note(f"The $p$-series theorem proves that this series <b>{verdict}</b>.")
        else:
            kind = "absolutely" if parameter > 1 else "conditionally"
            show_note(f"Leibniz gives convergence for every $p>0$; here it converges <b>{kind}</b>.")


partial_family.observe(configure_partial_parameter, names="value")
partial_button.on_click(run_partial_explorer)
configure_partial_parameter()
display(widgets.VBox([partial_family, partial_parameter, partial_N, partial_button, partial_output]))
run_partial_explorer()

## Cell 3 — A finite-window portrait of the Cauchy criterion

A series converges if and only if for every $\varepsilon>0$ there is an $N_0$ such that

$$
\left|\sum_{k=n}^{m}a_k\right|<\varepsilon
\qquad (m\ge n\ge N_0).
$$

For a fixed computational horizon $M$, define the finite-window envelope

$$
E_n(M)=\max_{n\le m\le M}\left|\sum_{k=n}^{m}a_k\right|.
$$

The graph below shows $E_n(M)$. A finite window can expose failure or illustrate tail decay, but it cannot by itself prove the universal Cauchy condition.

In [ ]:
cauchy_family = widgets.Dropdown(
    options=[
        ("Geometric: (0.65)^(n-1)", "geometric"),
        ("p-series: 1/n^p", "p_series"),
        ("Harmonic: 1/n", "harmonic"),
        ("Alternating harmonic", "alternating"),
        ("Telescoping: 1/[n(n+1)]", "telescoping"),
    ],
    value="alternating",
    description="Family:",
    layout=widgets.Layout(width="370px"),
)
cauchy_p = widgets.FloatSlider(
    value=2.0, min=0.5, max=3.0, step=0.1, description="p:",
    continuous_update=False, layout=widgets.Layout(width="370px")
)
cauchy_start = widgets.IntSlider(
    value=25, min=1, max=250, step=1, description="$n_0$:",
    continuous_update=False, layout=widgets.Layout(width="370px")
)
cauchy_M = widgets.IntSlider(
    value=300, min=50, max=800, step=25, description="Horizon M:",
    continuous_update=False, layout=widgets.Layout(width="370px")
)
cauchy_eps = widgets.FloatLogSlider(
    value=0.05, base=10, min=-4, max=0, step=0.1, description="$\\varepsilon$:",
    continuous_update=False, readout_format=".4f", layout=widgets.Layout(width="370px")
)
cauchy_button = style_button(widgets.Button(description="Inspect tails"), "search")
cauchy_output = widgets.Output()


def cauchy_terms(family, n, p):
    n = np.asarray(n, dtype=float)
    if family == "geometric":
        return 0.65 ** (n.astype(int) - 1)
    if family == "p_series":
        return n ** (-p)
    if family == "harmonic":
        return 1.0 / n
    if family == "alternating":
        return (-1.0) ** (n.astype(int) - 1) / n
    return 1.0 / (n * (n + 1.0))


def run_cauchy_lab(_=None):
    with cauchy_output:
        clear_output(wait=True)
        M = cauchy_M.value
        start = min(cauchy_start.value, M - 1)
        n = np.arange(1, M + 1)
        terms = cauchy_terms(cauchy_family.value, n, cauchy_p.value)
        partial = np.concatenate(([0.0], np.cumsum(terms)))

        # For each n, inspect every endpoint m between n and M.
        envelope = np.empty(M)
        for n0 in range(1, M + 1):
            envelope[n0 - 1] = np.max(np.abs(partial[n0:] - partial[n0 - 1]))

        selected_tail = envelope[start - 1]
        fig, ax = plt.subplots(figsize=(10.5, 4.2))
        ax.semilogy(n, np.maximum(envelope, 1e-16), color=COLORS["purple"], lw=2)
        ax.axhline(cauchy_eps.value, color=COLORS["red"], ls="--", label=r"chosen $\varepsilon$")
        ax.axvline(start, color=COLORS["gray"], ls=":", label=rf"$n_0={start}$")
        ax.set_xlabel("starting index $n$")
        ax.set_ylabel(r"$E_n(M)$")
        ax.set_title("Largest observed finite tail from $n$ to the horizon $M$")
        ax.legend()
        plt.show()

        print(f"E_{start}({M}) = {selected_tail:.6g}")
        relation = "below" if selected_tail < cauchy_eps.value else "not below"
        print(f"This observed envelope is {relation} epsilon = {cauchy_eps.value:.6g}.")
        show_note(
            "Keep $n$ fixed and increase $M$. Convergence requires control for <i>every</i> later endpoint, not only the displayed horizon."
        )


def configure_cauchy(change=None):
    cauchy_p.disabled = cauchy_family.value != "p_series"


cauchy_family.observe(configure_cauchy, names="value")
cauchy_button.on_click(run_cauchy_lab)
configure_cauchy()
display(widgets.VBox([cauchy_family, cauchy_p, cauchy_start, cauchy_M, cauchy_eps, cauchy_button, cauchy_output]))

## Cell 4 — Ratio and root diagnostics: useful, but sometimes inconclusive

For a series of nonzero terms, the numerical quantities

$$
Q_n=\left|\frac{a_{n+1}}{a_n}\right|,
\qquad
P_n=|a_n|^{1/n}
$$

approximate the ratio and root tests. A limiting value below $1$ proves absolute convergence. The value $1$ is **inconclusive**: both $\sum 1/n$ and $\sum1/n^2$ give the same limiting diagnostic. This laboratory deliberately includes examples where another test is needed.

In [ ]:
diagnostic_family = widgets.Dropdown(
    options=[
        ("p-series: 1/n^p", "p_series"),
        ("Polynomial times geometric: n^2/2^n", "n2_geometric"),
        ("Logarithmic p-series: 1/[n(log n)^p]", "log_p"),
        ("Log-only decay: 1/(log n)^p", "log_only"),
        ("Small-angle term: sin(1/n)^3", "sin_cube"),
    ],
    value="p_series",
    description="Family:",
    layout=widgets.Layout(width="430px"),
)
diagnostic_p = widgets.FloatSlider(
    value=2.0, min=0.25, max=3.0, step=0.05, description="p:",
    continuous_update=False, layout=widgets.Layout(width="430px")
)
diagnostic_N = widgets.IntSlider(
    value=400, min=30, max=800, step=10, description="N:",
    continuous_update=False, layout=widgets.Layout(width="430px")
)
diagnostic_button = style_button(widgets.Button(description="Run diagnostics"), "stethoscope")
diagnostic_output = widgets.Output()


def diagnostic_terms(family, n, p):
    n = np.asarray(n, dtype=float)
    if family == "p_series":
        return n ** (-p)
    if family == "n2_geometric":
        return n**2 / (2.0**n)
    if family == "log_p":
        return 1.0 / (n * np.log(n) ** p)
    if family == "log_only":
        return 1.0 / np.log(n) ** p
    return np.sin(1.0 / n) ** 3


def run_diagnostics(_=None):
    with diagnostic_output:
        clear_output(wait=True)
        family = diagnostic_family.value
        p = diagnostic_p.value
        N = diagnostic_N.value
        n = np.arange(2, N + 2)
        a = diagnostic_terms(family, n, p)
        ratios = np.abs(a[1:] / a[:-1])
        roots = np.exp(np.log(np.abs(a[:-1])) / n[:-1])

        tail_count = min(100, len(ratios))
        ratio_estimate = float(np.median(ratios[-tail_count:]))
        root_estimate = float(np.median(roots[-tail_count:]))

        fig, ax = plt.subplots(figsize=(10.5, 4.2))
        ax.plot(n[:-1], ratios, label=r"$|a_{n+1}/a_n|$", color=COLORS["blue"])
        ax.plot(n[:-1], roots, label=r"$|a_n|^{1/n}$", color=COLORS["orange"])
        ax.axhline(1, color=COLORS["red"], ls="--", lw=1.2, label="critical value 1")
        ax.set_ylim(bottom=max(0, min(np.min(ratios), np.min(roots)) - 0.08), top=min(1.5, max(np.max(ratios), np.max(roots), 1.05)))
        ax.set_xlabel("$n$")
        ax.set_title("Ratio and root diagnostics")
        ax.legend()
        plt.show()

        print(f"Median of the last {tail_count} ratio values: {ratio_estimate:.8f}")
        print(f"Median of the last {tail_count} root values:  {root_estimate:.8f}")

        if family == "p_series":
            verdict = "converges" if p > 1 else "diverges"
            explanation = f"Both diagnostics tend to 1. The p-series theorem, not these tests, proves that the series {verdict}."
        elif family == "n2_geometric":
            explanation = "Both diagnostics tend to 1/2, so either test proves absolute convergence."
        elif family == "log_p":
            verdict = "converges" if p > 1 else "diverges"
            explanation = f"The diagnostics tend to 1. Condensation or the integral test proves that the series {verdict}."
        elif family == "log_only":
            explanation = "The series diverges for every p: condensation produces terms that do not even approach zero."
        else:
            explanation = "Since sin(1/n)^3 is asymptotic to 1/n^3, limit comparison proves convergence."
        show_note(explanation)


def configure_diagnostic(change=None):
    diagnostic_p.disabled = diagnostic_family.value not in {"p_series", "log_p", "log_only"}


diagnostic_family.observe(configure_diagnostic, names="value")
diagnostic_button.on_click(run_diagnostics)
configure_diagnostic()
display(widgets.VBox([diagnostic_family, diagnostic_p, diagnostic_N, diagnostic_button, diagnostic_output]))

## Cell 5 — Integral-test bounds as certified numerical intervals

If $f$ is positive and decreasing and $a_n=f(n)$, the integral test gives

$$
\int_{N+1}^{\infty}f(x)\,dx
\le R_N
\le \int_N^{\infty}f(x)\,dx,
\qquad
R_N=\sum_{n=N+1}^{\infty}a_n.
$$

For $f(x)=x^{-p}$ with $p>1$,

$$
\frac{(N+1)^{1-p}}{p-1}\le R_N\le\frac{N^{1-p}}{p-1}.
$$

Thus a finite computation produces a rigorous interval containing the infinite sum.

In [ ]:
integral_p = widgets.FloatSlider(
    value=2.0, min=1.05, max=5.0, step=0.05, description="p:",
    continuous_update=False, readout_format=".2f", layout=widgets.Layout(width="390px")
)
integral_N = widgets.IntSlider(
    value=20, min=1, max=5000, step=1, description="N:",
    continuous_update=False, layout=widgets.Layout(width="390px")
)
integral_button = style_button(widgets.Button(description="Certify the sum"), "check-circle")
integral_output = widgets.Output()


def run_integral_bounds(_=None):
    with integral_output:
        clear_output(wait=True)
        p = integral_p.value
        N = integral_N.value
        n = np.arange(1, N + 1, dtype=float)
        S_N = float(np.sum(n ** (-p)))
        lower_tail = (N + 1.0) ** (1.0 - p) / (p - 1.0)
        upper_tail = N ** (1.0 - p) / (p - 1.0)
        lower_sum = S_N + lower_tail
        upper_sum = S_N + upper_tail
        exact = float(hurwitz_zeta(p, 1.0))

        display(Markdown(rf"### Certified enclosure for $\sum_{{n=1}}^\infty n^{{-{p:.2f}}}$"))
        print(f"S_N                         = {S_N:.12f}")
        print(f"Lower bound for the tail    = {lower_tail:.6e}")
        print(f"Upper bound for the tail    = {upper_tail:.6e}")
        print(f"Certified interval for sum  = [{lower_sum:.12f}, {upper_sum:.12f}]")
        print(f"Reference zeta value         = {exact:.12f}")
        print(f"Interval width               = {upper_sum - lower_sum:.6e}")

        sample_N = np.unique(np.geomspace(1, max(2, N), 120).astype(int))
        widths = sample_N ** (1.0 - p) / (p - 1.0) - (sample_N + 1.0) ** (1.0 - p) / (p - 1.0)
        fig, ax = plt.subplots(figsize=(9.5, 3.8))
        ax.loglog(sample_N, widths, color=COLORS["green"], lw=2)
        ax.scatter([N], [upper_sum - lower_sum], color=COLORS["red"], zorder=3)
        ax.set_xlabel("$N$")
        ax.set_ylabel("width of certified interval")
        ax.set_title("How the integral-test enclosure narrows")
        plt.show()


integral_button.on_click(run_integral_bounds)
display(widgets.VBox([integral_p, integral_N, integral_button, integral_output]))

## Cell 6 — Cauchy condensation with an arbitrary base

If $(a_n)$ is nonnegative and non-increasing and $q\ge2$ is an integer, then

$$
\sum_{n=1}^{\infty}a_n
\quad\text{and}\quad
\sum_{k=0}^{\infty}q^k a_{q^k}
$$

have the same convergence behaviour. For

$$
a_n=\frac{1}{n(\log n)^\alpha},
$$

condensation produces a constant multiple of $\sum k^{-\alpha}$. The third family below is the chapter's example showing that the base may affect the usefulness of the **ratio test**, although it cannot affect convergence itself.

In [ ]:
condensation_family = widgets.Dropdown(
    options=[
        ("p-series: 1/n^p", "p_series"),
        ("Logarithmic p-series: 1/[n(log n)^p]", "log_p"),
        ("Chapter example: base-sensitive ratio diagnostic", "base_example"),
    ],
    value="log_p",
    description="Family:",
    layout=widgets.Layout(width="470px"),
)
condensation_p = widgets.FloatSlider(
    value=1.0, min=0.25, max=3.0, step=0.05, description="p:",
    continuous_update=False, layout=widgets.Layout(width="470px")
)
condensation_q = widgets.IntSlider(
    value=2, min=2, max=6, step=1, description="Base q:",
    continuous_update=False, layout=widgets.Layout(width="470px")
)
condensation_K = widgets.IntSlider(
    value=14, min=4, max=18, step=1, description="Blocks K:",
    continuous_update=False, layout=widgets.Layout(width="470px")
)
condensation_button = style_button(widgets.Button(description="Condense series"), "compress")
condensation_output = widgets.Output()


def condensed_values(family, p, q, K):
    # The logarithmic family starts at k=1 because log(1)=0.
    k = np.arange(1, K + 1) if family == "log_p" else np.arange(0, K + 1)
    n = np.array([float(q) ** int(j) for j in k])
    if family == "p_series":
        a = n ** (-p)
    elif family == "log_p":
        a = 1.0 / (n * np.log(n) ** p)
    else:
        block = np.floor(np.log(n) / np.log(3.0) + 1e-12)
        a = 1.0 / (3.0**block * np.sqrt(n))
    return k, (q ** k) * a


def run_condensation(_=None):
    with condensation_output:
        clear_output(wait=True)
        family = condensation_family.value
        p = condensation_p.value
        q = condensation_q.value
        K = condensation_K.value
        k, b = condensed_values(family, p, q, K)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.0))
        axes[0].semilogy(k, b, marker="o", color=COLORS["purple"])
        axes[0].set_xlabel("block index $k$")
        axes[0].set_ylabel(r"$q^k a_{q^k}$")
        axes[0].set_title("Condensed terms")
        axes[1].plot(k, np.cumsum(b), marker="o", color=COLORS["blue"])
        axes[1].set_xlabel("$K$")
        axes[1].set_ylabel("condensed partial sum")
        axes[1].set_title("Condensed partial sums")
        plt.tight_layout()
        plt.show()

        if len(b) > 1:
            ratios = b[1:] / b[:-1]
            print("Last condensed ratios:", np.round(ratios[-min(8, len(ratios)):], 6))

        if family == "p_series":
            verdict = "converges" if p > 1 else "diverges"
            show_note(f"Here $q^k a_{{q^k}}=q^{{k(1-p)}}$. It is geometric, so the original series <b>{verdict}</b>.")
        elif family == "log_p":
            verdict = "converges" if p > 1 else "diverges"
            show_note(f"The condensed term is $1/[k^p(\\log q)^p]$. The original series therefore <b>{verdict}</b>.")
        else:
            if q == 3:
                show_note("For $q=3$, the condensed series is geometric with ratio $3^{-1/2}$; the ratio test is decisive.")
            elif q == 2:
                show_note("For $q=2$, the consecutive ratios oscillate across 1, so the ratio test is inconclusive; the root test still proves convergence.")
            else:
                show_note("Convergence is independent of the base. Inspect how the numerical ratio pattern changes with $q$.")


def configure_condensation(change=None):
    condensation_p.disabled = condensation_family.value == "base_example"


condensation_family.observe(configure_condensation, names="value")
condensation_button.on_click(run_condensation)
configure_condensation()
display(widgets.VBox([condensation_family, condensation_p, condensation_q, condensation_K, condensation_button, condensation_output]))

## Cell 7 — Refined tests near the critical ratio $1$

Kummer's general diagnostic uses an auxiliary positive sequence $(c_n)$:

$$
K_n=c_n-c_{n+1}\frac{a_{n+1}}{a_n}.
$$

A positive lower bound for $K_n$ proves convergence. With the additional condition $\sum1/c_n=\infty$, an eventual inequality $K_n\le0$ proves divergence. Choosing $c_n=n-1$ produces Raabe's quantity

$$
R_n=n\left(1-\frac{a_{n+1}}{a_n}\right).
$$

A limit above $1$ yields convergence and a limit below $1$ yields divergence. At the borderline $R_n\to1$, Bertrand introduces

$$
B_n=(\log n)(R_n-1).
$$

The controls below show why increasingly delicate asymptotic information is needed at a critical boundary.

In [ ]:
refined_family = widgets.Dropdown(
    options=[
        ("p-series: 1/n^p", "p_series"),
        ("Logarithmic p-series: 1/[n(log n)^p]", "log_p"),
        ("Product a_n = product (3k-2)/(3k)", "product_three"),
        ("Squared odd/even product", "odd_even_squared"),
    ],
    value="log_p",
    description="Family:",
    layout=widgets.Layout(width="470px"),
)
refined_p = widgets.FloatSlider(
    value=2.0, min=0.25, max=3.0, step=0.05, description="p:",
    continuous_update=False, layout=widgets.Layout(width="470px")
)
refined_N = widgets.IntSlider(
    value=1000, min=30, max=5000, step=10, description="N:",
    continuous_update=False, layout=widgets.Layout(width="470px")
)
refined_button = style_button(widgets.Button(description="Evaluate tests"), "microscope")
refined_output = widgets.Output()


def refined_ratio(family, n, p):
    n = np.asarray(n, dtype=float)
    if family == "p_series":
        return (n / (n + 1.0)) ** p
    if family == "log_p":
        return (n * np.log(n) ** p) / ((n + 1.0) * np.log(n + 1.0) ** p)
    if family == "product_three":
        return (3.0 * n + 1.0) / (3.0 * n + 3.0)
    return ((2.0 * n + 1.0) / (2.0 * n + 2.0)) ** 2


def run_refined_tests(_=None):
    with refined_output:
        clear_output(wait=True)
        family = refined_family.value
        p = refined_p.value
        N = refined_N.value
        n = np.arange(2, N + 1, dtype=float)
        ratio = refined_ratio(family, n, p)
        R = n * (1.0 - ratio)
        B = np.log(n) * (R - 1.0)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.0))
        axes[0].plot(n, R, color=COLORS["blue"], lw=2)
        axes[0].axhline(1, color=COLORS["red"], ls="--", label="Raabe boundary")
        axes[0].set_title(r"Raabe quantity $R_n$")
        axes[0].set_xlabel("$n$")
        axes[0].legend()
        axes[1].plot(n, B, color=COLORS["orange"], lw=2)
        axes[1].axhline(1, color=COLORS["red"], ls="--", label="Bertrand boundary")
        axes[1].set_title(r"Bertrand quantity $B_n$")
        axes[1].set_xlabel("$n$")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        print(f"R_N = {R[-1]:.8f}")
        print(f"B_N = {B[-1]:.8f}")

        if family == "p_series":
            verdict = "converges" if p > 1 else ("diverges" if p < 1 else "diverges, although Raabe is inconclusive")
            show_note(f"$R_n\\to p$. Therefore the series {verdict}.")
        elif family == "log_p":
            verdict = "converges" if p > 1 else "diverges"
            show_note(f"$R_n\\to1$ but $B_n\\to p$. Bertrand's test proves that the series <b>{verdict}</b>.")
        elif family == "product_three":
            show_note("$R_n\\to2/3<1$, so Raabe's test proves divergence; the ordinary ratio test was inconclusive.")
        else:
            show_note("$R_n\\to1$ and $B_n\\to0<1$, so Bertrand proves divergence; Gauss's controlled expansion gives the same conclusion.")


def configure_refined(change=None):
    refined_p.disabled = refined_family.value not in {"p_series", "log_p"}


refined_family.observe(configure_refined, names="value")
refined_button.on_click(run_refined_tests)
configure_refined()
display(widgets.VBox([refined_family, refined_p, refined_N, refined_button, refined_output]))

## Cell 8 — Cauchy products as discrete convolution

Given $\sum_{n=0}^{\infty}a_n$ and $\sum_{n=0}^{\infty}b_n$, their Cauchy product has coefficients

$$
c_n=\sum_{k=0}^{n}a_kb_{n-k}.
$$

If both series converge absolutely, then

$$
\sum_{n=0}^{\infty}c_n
=\left(\sum_{n=0}^{\infty}a_n\right)
 \left(\sum_{n=0}^{\infty}b_n\right).
$$

For geometric inputs $a_n=r^n$ and $b_n=s^n$, the widgets expose the convolution coefficients and the convergence of their partial sums to $[(1-r)(1-s)]^{-1}$.

In [ ]:
cauchy_r = widgets.FloatSlider(
    value=0.40, min=-0.85, max=0.85, step=0.05, description="r:",
    continuous_update=False, layout=widgets.Layout(width="390px")
)
cauchy_s = widgets.FloatSlider(
    value=0.65, min=-0.85, max=0.85, step=0.05, description="s:",
    continuous_update=False, layout=widgets.Layout(width="390px")
)
product_N = widgets.IntSlider(
    value=25, min=3, max=100, step=1, description="N:",
    continuous_update=False, layout=widgets.Layout(width="390px")
)
product_button = style_button(widgets.Button(description="Compute product"), "project-diagram")
product_output = widgets.Output()


def run_cauchy_product(_=None):
    with product_output:
        clear_output(wait=True)
        r, s, N = cauchy_r.value, cauchy_s.value, product_N.value
        n = np.arange(N + 1)
        a = r**n
        b = s**n
        c = np.convolve(a, b)[: N + 1]
        C = np.cumsum(c)
        exact = 1.0 / ((1.0 - r) * (1.0 - s))
        A_N = np.sum(a)
        B_N = np.sum(b)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.0))
        axes[0].stem(n, c, linefmt=COLORS["purple"], markerfmt="o", basefmt=" ")
        axes[0].set_title("Cauchy-product coefficients $c_n$")
        axes[0].set_xlabel("$n$")
        axes[1].plot(n, C, color=COLORS["blue"], lw=2, label=r"$\sum_{k=0}^n c_k$")
        axes[1].axhline(exact, color=COLORS["red"], ls="--", label="exact infinite product")
        axes[1].set_title("Partial sums of the product series")
        axes[1].set_xlabel("$n$")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        print(f"Product-series partial sum C_N = {C[-1]:.12f}")
        print(f"Exact infinite product          = {exact:.12f}")
        print(f"|C_N - exact|                  = {abs(C[-1] - exact):.3e}")
        print(f"A_N B_N                        = {A_N * B_N:.12f}")
        show_note("In general $C_N$ is not equal to $A_NB_N$: the latter also contains convolution terms of total degree greater than $N$.")


product_button.on_click(run_cauchy_product)
display(widgets.VBox([cauchy_r, cauchy_s, product_N, product_button, product_output]))

## Cell 9 — Abel's limit theorem: approaching the boundary radially

If $\sum_{n=0}^{\infty}a_n=A$, Abel's limit theorem states that

$$
\lim_{r\uparrow1}\sum_{n=0}^{\infty}a_nr^n=A.
$$

For each fixed $0\le r<1$, the factor $r^n$ regularizes the series. The theorem still requires convergence of the original series. The optional Grandi example has an Abel limit but is not classically convergent, so it illustrates why one must not reverse the theorem.

In [ ]:
abel_family = widgets.Dropdown(
    options=[
        ("Alternating harmonic coefficients", "alternating_harmonic"),
        ("Geometric coefficients: a_n=(0.7)^n", "geometric"),
        ("Grandi coefficients: a_n=(-1)^n (outside theorem)", "grandi"),
    ],
    value="alternating_harmonic",
    description="Coefficients:",
    layout=widgets.Layout(width="500px"),
)
abel_r = widgets.FloatSlider(
    value=0.90, min=0.0, max=0.999, step=0.001, description="r:",
    continuous_update=False, readout_format=".3f", layout=widgets.Layout(width="500px")
)
abel_N = widgets.IntSlider(
    value=300, min=10, max=5000, step=10, description="Terms N:",
    continuous_update=False, layout=widgets.Layout(width="500px")
)
abel_button = style_button(widgets.Button(description="Approach boundary"), "arrow-right")
abel_output = widgets.Output()


def abel_data(family, r, N):
    n = np.arange(N + 1)
    if family == "alternating_harmonic":
        coefficients = (-1.0) ** n / (n + 1.0)
        radial_exact = math.log1p(r) / r if r > 0 else 1.0
        boundary_value = math.log(2.0)
    elif family == "geometric":
        coefficients = 0.7**n
        radial_exact = 1.0 / (1.0 - 0.7 * r)
        boundary_value = 1.0 / 0.3
    else:
        coefficients = (-1.0) ** n
        radial_exact = 1.0 / (1.0 + r)
        boundary_value = 0.5
    radial_partial = float(np.sum(coefficients * r**n))
    return radial_partial, radial_exact, boundary_value


def run_abel_lab(_=None):
    with abel_output:
        clear_output(wait=True)
        family, r, N = abel_family.value, abel_r.value, abel_N.value
        partial, exact, boundary = abel_data(family, r, N)
        r_grid = np.linspace(0.0, 0.999, 300)

        if family == "alternating_harmonic":
            curve = np.ones_like(r_grid)
            positive = r_grid > 0
            curve[positive] = np.log1p(r_grid[positive]) / r_grid[positive]
        elif family == "geometric":
            curve = 1.0 / (1.0 - 0.7 * r_grid)
        else:
            curve = 1.0 / (1.0 + r_grid)

        fig, ax = plt.subplots(figsize=(10.2, 4.1))
        ax.plot(r_grid, curve, color=COLORS["blue"], lw=2, label=r"exact $\sum a_nr^n$")
        ax.scatter([r], [exact], color=COLORS["red"], s=55, zorder=3, label="selected r")
        ax.axhline(boundary, color=COLORS["green"], ls="--", label="boundary value")
        ax.set_xlabel("$r$")
        ax.set_title("Radial approach to $r=1$")
        ax.legend()
        plt.show()

        print(f"Truncated radial sum through n={N}: {partial:.12f}")
        print(f"Exact radial sum at r={r:.3f}:      {exact:.12f}")
        print(f"Truncation error:                   {abs(partial - exact):.3e}")
        print(f"Limit as r approaches 1:            {boundary:.12f}")
        if family == "grandi":
            show_note("The Grandi series does not converge classically. Its radial limit $1/2$ is an Abel sum, not an application of Abel's limit theorem.", "#fff7ed")
        else:
            show_note("The original coefficient series converges, so Abel's theorem identifies the boundary limit with its ordinary sum.")


abel_button.on_click(run_abel_lab)
display(widgets.VBox([abel_family, abel_r, abel_N, abel_button, abel_output]))

## Cell 10 — Dirichlet cancellation and summation by parts

Dirichlet's test separates oscillation from decay. If

$$
A_N=\sum_{n=1}^{N}a_n
$$

is bounded and $b_n\downarrow0$, then $\sum a_nb_n$ converges. For $a_n=\sin(n\theta)$ with $\theta\notin2\pi\mathbb Z$,

$$
\left|\sum_{n=1}^{N}e^{in\theta}\right|
\le \frac{2}{|1-e^{i\theta}|}.
$$

Choose $b_n=n^{-\beta}$ with $\beta>0$. The left plot tests bounded oscillation; the right plot displays convergence created by multiplying that oscillation by a decreasing factor.

In [ ]:
dirichlet_theta = widgets.FloatSlider(
    value=1.20, min=0.10, max=6.10, step=0.05, description="$\\theta$:",
    continuous_update=False, layout=widgets.Layout(width="410px")
)
dirichlet_beta = widgets.FloatSlider(
    value=1.0, min=0.20, max=2.0, step=0.05, description="$\\beta$:",
    continuous_update=False, layout=widgets.Layout(width="410px")
)
dirichlet_N = widgets.IntSlider(
    value=400, min=20, max=3000, step=20, description="N:",
    continuous_update=False, layout=widgets.Layout(width="410px")
)
dirichlet_button = style_button(widgets.Button(description="Show cancellation"), "wave-square")
dirichlet_output = widgets.Output()


def run_dirichlet(_=None):
    with dirichlet_output:
        clear_output(wait=True)
        theta, beta, N = dirichlet_theta.value, dirichlet_beta.value, dirichlet_N.value
        n = np.arange(1, N + 1, dtype=float)
        oscillation = np.sin(n * theta)
        A = np.cumsum(oscillation)
        b = n ** (-beta)
        weighted_sums = np.cumsum(oscillation * b)
        complex_bound = 2.0 / abs(1.0 - np.exp(1j * theta))

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.0))
        axes[0].plot(n, A, color=COLORS["purple"], lw=1.4)
        axes[0].axhline(complex_bound, color=COLORS["red"], ls="--", lw=1)
        axes[0].axhline(-complex_bound, color=COLORS["red"], ls="--", lw=1)
        axes[0].set_title(r"Bounded sums $A_N=\sum_{n\leq N}\sin(n\theta)$")
        axes[0].set_xlabel("$N$")
        axes[1].plot(n, weighted_sums, color=COLORS["blue"], lw=1.8)
        axes[1].set_title(r"Partial sums of $\sum \sin(n\theta)/n^\beta$")
        axes[1].set_xlabel("$N$")
        plt.tight_layout()
        plt.show()

        print(f"Observed max |A_N|             = {np.max(np.abs(A)):.8f}")
        print(f"Valid complex-sum upper bound = {complex_bound:.8f}")
        print(f"Weighted partial sum S_N      = {weighted_sums[-1]:.10f}")
        show_note("The plot illustrates the hypotheses. The proof of convergence uses Abel summation to control every remote tail uniformly.")


dirichlet_button.on_click(run_dirichlet)
display(widgets.VBox([dirichlet_theta, dirichlet_beta, dirichlet_N, dirichlet_button, dirichlet_output]))

## Cell 11 — Alternating series: trapping and a rigorous error bound

If $a_n\downarrow0$, Leibniz's theorem states that

$$
\sum_{n=1}^{\infty}(-1)^{n-1}a_n
$$

converges and its sum $S$ lies between two consecutive partial sums. Consequently,

$$
|S-S_N|\le a_{N+1}.
$$

For $a_n=n^{-p}$ with $p>0$, the series always converges. It is absolutely convergent exactly when $p>1$.

In [ ]:
alternating_p = widgets.FloatSlider(
    value=1.0, min=0.20, max=3.0, step=0.05, description="p:",
    continuous_update=False, layout=widgets.Layout(width="410px")
)
alternating_N = widgets.IntSlider(
    value=20, min=1, max=1000, step=1, description="N:",
    continuous_update=False, layout=widgets.Layout(width="410px")
)
alternating_button = style_button(widgets.Button(description="Trap the sum"), "arrows-alt-v")
alternating_output = widgets.Output()


def run_alternating(_=None):
    with alternating_output:
        clear_output(wait=True)
        p, N = alternating_p.value, alternating_N.value
        plot_N = max(N + 1, min(300, max(40, N)))
        n = np.arange(1, plot_N + 1, dtype=float)
        terms = (-1.0) ** (n.astype(int) - 1) * n ** (-p)
        sums = np.cumsum(terms)
        S_N = sums[N - 1]
        S_next = sums[N]
        lower, upper = sorted([S_N, S_next])
        bound = (N + 1.0) ** (-p)

        fig, ax = plt.subplots(figsize=(10.5, 4.2))
        indices = np.arange(1, plot_N + 1)
        ax.plot(indices[::2], sums[::2], "o-", ms=3, color=COLORS["orange"], label="odd partial sums")
        ax.plot(indices[1::2], sums[1::2], "o-", ms=3, color=COLORS["blue"], label="even partial sums")
        ax.axhspan(lower, upper, color=COLORS["green"], alpha=0.16, label="rigorous interval from $S_N,S_{N+1}$")
        ax.set_xlabel("$N$")
        ax.set_title("Even and odd partial sums trap the limit")
        ax.legend()
        plt.show()

        print(f"S_N                         = {S_N:.12f}")
        print(f"S_(N+1)                     = {S_next:.12f}")
        print(f"Rigorous interval for S     = [{lower:.12f}, {upper:.12f}]")
        print(f"Leibniz error bound         = {bound:.6e}")
        if abs(p - 1.0) < 1e-12:
            print(f"Exact value log(2)          = {math.log(2):.12f}")
            print(f"Actual error                = {abs(math.log(2) - S_N):.6e}")
        kind = "absolute" if p > 1 else "conditional"
        show_note(f"For this value of $p$, convergence is <b>{kind}</b>.")


alternating_button.on_click(run_alternating)
display(widgets.VBox([alternating_p, alternating_N, alternating_button, alternating_output]))

## Cell 12 — Riemann rearrangement: steering a conditional series

In a conditionally convergent real series,

$$
\sum a_n^+=+\infty,
\qquad
\sum a_n^-=+\infty.
$$

To target a prescribed $L\in\mathbb R$, add unused positive terms until the partial sum crosses above $L$, then unused negative terms until it crosses below $L$, and repeat. Because the terms tend to zero, the overshoots shrink. This cell applies the construction to

$$
1-\frac12+\frac13-\frac14+\cdots.
$$

In [ ]:
rearrange_target = widgets.FloatSlider(
    value=1.0, min=-1.5, max=2.0, step=0.05, description="Target L:",
    continuous_update=False, layout=widgets.Layout(width="420px")
)
rearrange_terms = widgets.IntSlider(
    value=1200, min=50, max=5000, step=50, description="Terms used:",
    continuous_update=False, layout=widgets.Layout(width="420px")
)
rearrange_button = style_button(widgets.Button(description="Build rearrangement"), "random")
rearrange_output = widgets.Output()


def alternating_harmonic_rearrangement(target, max_terms):
    """Return a finite prefix of Riemann's target-crossing rearrangement."""
    positive_index = 1  # term 1/(2j-1)
    negative_index = 1  # term -1/(2j)
    total = 0.0
    sums, original_indices = [], []

    while len(sums) < max_terms:
        while total <= target and len(sums) < max_terms:
            original_n = 2 * positive_index - 1
            total += 1.0 / original_n
            sums.append(total)
            original_indices.append(original_n)
            positive_index += 1
        while total >= target and len(sums) < max_terms:
            original_n = 2 * negative_index
            total -= 1.0 / original_n
            sums.append(total)
            original_indices.append(original_n)
            negative_index += 1
    return np.asarray(sums), original_indices


def run_rearrangement(_=None):
    with rearrange_output:
        clear_output(wait=True)
        target, count = rearrange_target.value, rearrange_terms.value
        sums, order = alternating_harmonic_rearrangement(target, count)
        n = np.arange(1, len(sums) + 1)

        fig, ax = plt.subplots(figsize=(10.5, 4.2))
        ax.plot(n, sums, color=COLORS["purple"], lw=1.2)
        ax.axhline(target, color=COLORS["red"], ls="--", lw=1.5, label=f"target L = {target:.2f}")
        ax.set_xlabel("number of terms in the rearrangement")
        ax.set_ylabel("partial sum")
        ax.set_title("Riemann target-crossing construction")
        ax.legend()
        plt.show()

        print("First 16 original indices in the new order:")
        print(order[:16])
        print(f"Final displayed partial sum = {sums[-1]:.10f}")
        print(f"Distance from target         = {abs(sums[-1] - target):.3e}")
        largest_recent_term = max(1.0 / order[-1], 1.0 / order[-2])
        print(f"Scale of the last terms      = {largest_recent_term:.3e}")
        show_note("This finite prefix illustrates the construction. The theorem proves that every original term is eventually used exactly once and that the crossings converge to $L$.")


rearrange_button.on_click(run_rearrangement)
display(widgets.VBox([rearrange_target, rearrange_terms, rearrange_button, rearrange_output]))

## Cell 13 — Interactive convergence-test selector

No single convergence test dominates all others. Before calculating a limit, inspect the structure of $a_n$ and verify the hypotheses. For example:

$$
\frac{a_{n+1}}{a_n}\to1
$$

is not a conclusion—it is a signal to change methods. Choose the visible structure below and press the button to obtain a theorem, its decisive hypothesis, and its main warning.

In [ ]:
test_cases = {
    "Terms do not approach zero": (
        "Term test",
        r"Check $\lim a_n$. If it is nonzero or does not exist, the series diverges.",
        r"The converse is false: $a_n\to0$ does not imply convergence."
    ),
    "Positive rational or algebraic terms": (
        "Comparison or limit comparison",
        r"Identify the dominant power and compare with $1/n^p$.",
        r"Limit comparison requires nonnegative terms and a finite positive comparison limit."
    ),
    "Factorials, exponentials, repeated products": (
        "Ratio test",
        r"Study $|a_{n+1}/a_n|$.",
        r"A limiting value equal to $1$ is inconclusive."
    ),
    "Terms resembling (u_n)^n": (
        "Root test",
        r"Study $|a_n|^{1/n}$, preferably through logarithms.",
        r"Again, the critical value $1$ is inconclusive."
    ),
    "Positive decreasing terms with logarithms": (
        "Condensation or integral test",
        r"Verify positivity and eventual monotonicity before grouping or integrating.",
        r"Do not apply condensation to an arbitrary oscillating sequence."
    ),
    "Quotient equals 1 + O(1/n)": (
        "Raabe, Bertrand, or Gauss",
        r"Compute the first meaningful correction to the quotient.",
        r"Critical constants require a finer expansion; numerical rounding is dangerous."
    ),
    "Oscillation with bounded partial sums": (
        "Dirichlet test",
        r"Show $A_N=\sum_{n\le N}a_n$ is bounded and $b_n\downarrow0$.",
        r"Bounded individual terms $a_n$ are not enough; their partial sums must be bounded."
    ),
    "Alternating decreasing magnitudes": (
        "Leibniz test",
        r"Verify $a_n\ge0$, $a_{n+1}\le a_n$, and $a_n\to0$.",
        r"The estimate $|R_N|\le a_{N+1}$ requires monotone magnitudes."
    ),
    "Order or multiplication matters": (
        "Absolute convergence / Cauchy-product theorems",
        r"Check $\sum|a_n|$ before rearranging terms or multiplying series.",
        r"Conditional convergence does not permit unrestricted rearrangement."
    ),
}

selector = widgets.Dropdown(
    options=list(test_cases.keys()),
    description="Structure:",
    layout=widgets.Layout(width="560px"),
)
selector_button = style_button(widgets.Button(description="Suggest a test"), "compass")
selector_output = widgets.Output()


def suggest_test(_=None):
    with selector_output:
        clear_output(wait=True)
        theorem, check, warning = test_cases[selector.value]
        display(Markdown(f"### Suggested method: **{theorem}**"))
        display(Markdown(f"**Decisive check.** {check}"))
        display(Markdown(f"**Warning.** {warning}"))


selector_button.on_click(suggest_test)
display(widgets.VBox([selector, selector_button, selector_output]))
suggest_test()

## Cell 14 — Self-check quiz with immediate feedback

A theorem is useful only when its hypotheses and conclusion are kept distinct. This quiz reviews the logical implications

$$
\sum a_n\text{ converges}\Longrightarrow a_n\to0,
\qquad
\sum|a_n|\text{ converges}\Longrightarrow\sum a_n\text{ converges},
$$

and the critical cases of the main tests. Answer one question at a time; the explanation appears only after you press **Check answer**.

In [ ]:
quiz_questions = [
    {
        "prompt": r"If $a_n\to0$, what can we conclude about $\sum a_n$?",
        "options": ["It converges", "It diverges", "No conclusion from this fact alone"],
        "answer": "No conclusion from this fact alone",
        "explanation": r"The condition is necessary but not sufficient: compare $\sum1/n$ with $\sum1/n^2$."
    },
    {
        "prompt": r"For which values of $p$ does $\sum_{n=1}^\infty n^{-p}$ converge?",
        "options": ["p > 0", "p > 1", "p >= 1"],
        "answer": "p > 1",
        "explanation": r"This is the $p$-series theorem. The boundary $p=1$ is the divergent harmonic series."
    },
    {
        "prompt": r"The ratio $|a_{n+1}/a_n|$ tends to $1$. What does the ratio test say?",
        "options": ["Absolute convergence", "Divergence", "The test is inconclusive"],
        "answer": "The test is inconclusive",
        "explanation": r"At the critical value $1$, use comparison, condensation, an integral, or a refined test."
    },
    {
        "prompt": r"Classify $\sum_{n=2}^\infty 1/[n(\log n)^2]$.",
        "options": ["Convergent", "Divergent", "Terms do not tend to zero"],
        "answer": "Convergent",
        "explanation": r"Condensation reduces it to a constant multiple of $\sum1/k^2$."
    },
    {
        "prompt": r"For a Leibniz series, which bound is guaranteed after $N$ terms?",
        "options": [r"$|R_N|\le a_N$", r"$|R_N|\le a_{N+1}$", r"$|R_N|\le a_{N+1}^2$"],
        "answer": r"$|R_N|\le a_{N+1}$",
        "explanation": r"The sum lies between $S_N$ and $S_{N+1}$, whose distance is $a_{N+1}$."
    },
    {
        "prompt": r"Which property makes every rearrangement preserve the sum?",
        "options": ["Termwise positivity only", "Conditional convergence", "Absolute convergence"],
        "answer": "Absolute convergence",
        "explanation": r"A conditionally convergent real series can be rearranged to any prescribed real value."
    },
]

quiz_state = {"index": 0, "score": 0, "answered": False}
quiz_prompt = widgets.HTMLMath()
quiz_choice = widgets.ToggleButtons(layout=widgets.Layout(width="100%"))
quiz_check = style_button(widgets.Button(description="Check answer"), "check")
quiz_next = widgets.Button(description="Next question", icon="arrow-right", disabled=True)
quiz_restart = widgets.Button(description="Restart quiz", icon="redo")
quiz_feedback = widgets.Output()
quiz_score = widgets.HTML()


def render_quiz_question():
    q = quiz_questions[quiz_state["index"]]
    quiz_prompt.value = f"<h4>Question {quiz_state['index'] + 1} of {len(quiz_questions)}</h4><p>{q['prompt']}</p>"
    quiz_choice.options = q["options"]
    quiz_choice.value = None
    quiz_state["answered"] = False
    quiz_check.disabled = False
    quiz_next.disabled = True
    quiz_score.value = f"<b>Score:</b> {quiz_state['score']} / {quiz_state['index']} completed"
    with quiz_feedback:
        clear_output()


def check_quiz_answer(_=None):
    if quiz_state["answered"]:
        return
    with quiz_feedback:
        clear_output(wait=True)
        if quiz_choice.value is None:
            display(Markdown("Please choose an answer first."))
            return
        q = quiz_questions[quiz_state["index"]]
        correct = quiz_choice.value == q["answer"]
        if correct:
            quiz_state["score"] += 1
            display(Markdown("### Correct"))
        else:
            display(Markdown(f"### Not yet — the correct answer is **{q['answer']}**"))
        display(Markdown(q["explanation"]))
    quiz_state["answered"] = True
    quiz_check.disabled = True
    quiz_next.disabled = False
    quiz_score.value = f"<b>Score:</b> {quiz_state['score']} / {quiz_state['index'] + 1} completed"


def next_quiz_question(_=None):
    if quiz_state["index"] < len(quiz_questions) - 1:
        quiz_state["index"] += 1
        render_quiz_question()
    else:
        with quiz_feedback:
            clear_output(wait=True)
            display(Markdown(f"## Quiz complete: {quiz_state['score']} / {len(quiz_questions)}"))
        quiz_next.disabled = True


def restart_quiz(_=None):
    quiz_state.update(index=0, score=0, answered=False)
    render_quiz_question()


quiz_check.on_click(check_quiz_answer)
quiz_next.on_click(next_quiz_question)
quiz_restart.on_click(restart_quiz)
render_quiz_question()
display(widgets.VBox([
    quiz_prompt,
    quiz_choice,
    widgets.HBox([quiz_check, quiz_next, quiz_restart]),
    quiz_score,
    quiz_feedback,
]))

## Cell 15 — Practice generator: choose, justify, then reveal

A rigorous classification should have the form

$$
\text{hypotheses}\quad\Longrightarrow\quad
\text{named theorem}\quad\Longrightarrow\quad
\text{conclusion}.
$$

Press **New problem**, decide which test is efficient, and write down the hypotheses before using the hint or solution buttons. The goal is method selection, not pattern matching.

In [ ]:
practice_bank = [
    {
        "problem": r"Classify $\displaystyle\sum_{n=1}^{\infty}\frac{n^2}{2^n}$.",
        "hint": r"Compute $a_{n+1}/a_n$ and simplify before taking the limit.",
        "solution": r"The ratio tends to $1/2<1$, so the ratio test proves absolute convergence."
    },
    {
        "problem": r"Classify $\displaystyle\sum_{n=2}^{\infty}\frac{1}{n\log n}$.",
        "hint": r"The ratio and root tests approach $1$. Try condensation or the integral test.",
        "solution": r"Condensation gives $2^k/[2^k\log(2^k)]=1/(k\log2)$, a divergent harmonic multiple."
    },
    {
        "problem": r"Classify $\displaystyle\sum_{n=1}^{\infty}\frac{(-1)^{n-1}n}{n^2+1}$.",
        "hint": r"First study the positive magnitudes, then test their absolute-value series.",
        "solution": r"The magnitudes decrease eventually to zero, so Leibniz gives convergence. They are asymptotic to $1/n$, so absolute convergence fails; the series is conditionally convergent."
    },
    {
        "problem": r"Classify $\displaystyle\sum_{n=1}^{\infty}\sin^3(1/n)$.",
        "hint": r"Use $\sin x/x\to1$ and compare with a familiar power.",
        "solution": r"The quotient $\sin^3(1/n)/(1/n^3)\to1$. Limit comparison with $\sum1/n^3$ proves convergence."
    },
    {
        "problem": r"Classify $\displaystyle\sum_{n=1}^{\infty}\frac{\sin(n\theta)}{n}$ for $\theta\notin2\pi\mathbb Z$.",
        "hint": r"Bound the partial sums of $e^{in\theta}$ using the finite geometric identity.",
        "solution": r"The partial sums of $\sin(n\theta)$ are bounded, while $1/n\downarrow0$. Dirichlet's test proves convergence."
    },
    {
        "problem": r"Can a rearrangement change the value of $\displaystyle\sum_{n=1}^{\infty}(-1)^n/n^2$?",
        "hint": r"Test absolute convergence before invoking any rearrangement theorem.",
        "solution": r"No. Since $\sum1/n^2$ converges, the series is absolutely convergent and every rearrangement has the same sum."
    },
    {
        "problem": r"Estimate the remainder of $\displaystyle\sum_{n=1}^{\infty}(-1)^{n-1}/n$ after $N=100$ terms.",
        "hint": r"Use the first omitted magnitude.",
        "solution": r"Leibniz gives $|R_{100}|\le1/101\approx9.901\times10^{-3}$."
    },
]

practice_rng = random.Random(2026)
practice_state = {"item": None}
practice_output = widgets.Output()
new_problem_button = style_button(widgets.Button(description="New problem"), "dice")
hint_button = widgets.Button(description="Show hint", icon="lightbulb")
solution_button = widgets.Button(description="Reveal solution", icon="eye")


def new_practice_problem(_=None):
    practice_state["item"] = practice_rng.choice(practice_bank)
    with practice_output:
        clear_output(wait=True)
        display(Markdown("### Problem"))
        display(Markdown(practice_state["item"]["problem"]))
        display(Markdown("_Before revealing anything: name the test and list every hypothesis._"))


def show_practice_hint(_=None):
    if practice_state["item"] is None:
        new_practice_problem()
    with practice_output:
        display(Markdown(f"**Hint.** {practice_state['item']['hint']}"))


def show_practice_solution(_=None):
    if practice_state["item"] is None:
        new_practice_problem()
    with practice_output:
        display(Markdown(f"**Solution.** {practice_state['item']['solution']}"))


new_problem_button.on_click(new_practice_problem)
hint_button.on_click(show_practice_hint)
solution_button.on_click(show_practice_solution)
display(widgets.VBox([
    widgets.HBox([new_problem_button, hint_button, solution_button]),
    practice_output,
]))
new_practice_problem()

## Suggested learning path and extensions

1. In the partial-sum laboratory, compare $p=0.9$, $p=1$, and $p=1.1$. Explain why a graph alone has difficulty distinguishing slow convergence from divergence.
2. In the Cauchy laboratory, keep the starting index fixed and increase the horizon. Record how the envelope changes for the harmonic and alternating harmonic series.
3. Use bases $q=2$ and $q=3$ in the chapter's condensation example. Explain precisely why the convergence conclusion is unchanged while the ratio-test diagnostic changes.
4. In the refined-test laboratory, compare Raabe's and Bertrand's quantities for $1/[n(\log n)^p]$ at $p=0.8$, $1$, and $1.2$.
5. Use the Dirichlet laboratory with small $\theta$. Relate the larger bound to the factor $|1-e^{i\theta}|^{-1}$.
6. Rearrange the alternating harmonic series toward $L=-1$, $L=1$, and $L=2$. Explain why slower-decaying terms make visible convergence slow.

### Final reminder

A plot can reveal structure, but a proof must still state the theorem, verify its hypotheses, and justify the conclusion. The central object remains

$$
S_N=\sum_{n=1}^{N}a_n,
$$

together with rigorous control of its remote tails.